# Qwen3.5-9B-Instruct 全参数微调（NPU 昇腾 8×910B + SFTTrainer）

## 环境说明
- **硬件：** 8×Ascend 910B NPU
- **平台：** ModelArts / 华为云
- **CANN 版本：** 8.5.0
- **模型：** Qwen/Qwen3.5-9B-Instruct
- **框架：** TRL SFTTrainer + DeepSpeed ZeRO2
- **训练方式：** 全参数微调（Full Fine-Tune）
- **特性：** 启用 packing（序列拼接）提升训练效率


In [ ]:
# ============================================================
# Cell 0: 安装依赖（首次运行执行，有缓存可跳过）
# ============================================================
# %%capture
# !pip install -i https://pypi.tuna.tsinghua.edu.cn/simple -U datasets accelerate peft trl tensorboard sentencepiece transformers

# DeepSpeed 必须锁定 0.16.9，0.17.4 不兼容昇腾 NPU
# !pip install deepspeed==0.16.9


In [ ]:
# ============================================================
# Cell 1: 导入依赖
# ============================================================
import os, sys, json, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch
import torch_npu           # 昇腾 NPU 必须导入
import transformers
from pprint import pprint
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTConfig, SFTTrainer        # 使用 TRL 的 SFTTrainer，自带 packing

print(f'torch version    = {torch.__version__}')
print(f'transformers     = {transformers.__version__}')


In [ ]:
# ============================================================
# Cell 2: 设备检测 — 昇腾 NPU
# ============================================================
device = torch.device('npu:0' if torch.npu.is_available() else 'cpu')
npu_cnt = torch.npu.device_count()
print(f'device          = {device}')
print(f'npu device cnt  = {npu_cnt}')
assert torch.npu.is_available(), 'NPU 不可用，请检查 torch_npu 安装'
print(f'检测到 {npu_cnt} 张 NPU 卡，准备工作完成')


In [ ]:
# ============================================================
# Cell 3: 路径配置 — 根据实际环境修改
# ============================================================
path_project = '/home/work/user-job-dir/fft_qwen3.5_9b'
path_data = os.path.join(path_project, 'data')
path_output = os.path.join(path_project, 'output')
os.makedirs(path_data, exist_ok=True)
os.makedirs(path_output, exist_ok=True)
print(f'项目路径: {path_project}')
print(f'数据路径: {path_data}')
print(f'输出路径: {path_output}')


## Step 1: 数据加载与预处理


In [ ]:
# ============================================================
# Cell 4: 加载训练数据（Alpaca 格式 parquet）
# ============================================================
filename = 'train-00000-of-00001-a09b74b3ef9c3b56.parquet'
dataset = load_dataset(
    path='parquet',
    data_files=os.path.join(path_data, filename),
    split='all')
print(f'原始数据量: {len(dataset)}')
print(f'数据字段: {dataset.column_names}')
pprint(dataset[0])


In [ ]:
# ============================================================
# Cell 5: 查看数据样例
# ============================================================
for i in range(3):
    print(f'\n{"="*60}')
    print(f'样例 {i}:')
    print(f'  指令: {dataset[i]["instruction"][:100]}...')
    print(f'  输入: {dataset[i]["input"][:100]}...')
    print(f'  输出: {dataset[i]["output"][:100]}...')


In [ ]:
# ============================================================
# Cell 6: 加载 tokenizer 用于数据格式化
# ============================================================
# Qwen3.5 tokenizer 自带特殊 token，不要手动 add_special_tokens
model_name_or_path = 'Qwen/Qwen3.5-9B'
tmp_tokenizer = AutoTokenizer.from_pretrained(
    model_name_or_path, trust_remote_code=True)
print(f'bos_token: {tmp_tokenizer.bos_token}')
print(f'eos_token: {tmp_tokenizer.eos_token}')
tmp_tokenizer.padding_side = 'right'


In [ ]:
# ============================================================
# Cell 7: 定义格式化函数（Alpaca -> chat_template）
# ============================================================
SYSTEM_PROMPT = '你是一个有帮助的助手。'
def format_chat_sample(sample, tokenizer):
    user_content = sample['instruction']
    if sample['input'] and sample['input'].strip():
        user_content += '\n' + sample['input']
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_content},
        {'role': 'assistant', 'content': sample['output']},
    ]
    sample['text'] = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False)
    return sample

# 测试格式化效果
test_sample = format_chat_sample(dataset[0], tmp_tokenizer)
print('格式化后的文本（前300字符）:')
print(test_sample['text'][:300])
print('\n...\n后200字符:')
print(test_sample['text'][-200:])


In [ ]:
# ============================================================
# Cell 8: 格式化数据集 & 划分训练/验证集
# ============================================================
dataset = dataset.map(
    lambda x: format_chat_sample(x, tmp_tokenizer),
    desc='正在格式化数据...')
dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset['train']
eval_dataset = dataset['test']
print(f'训练集: {len(train_dataset)} 条')
print(f'验证集: {len(eval_dataset)} 条')


## Step 2: 加载 Tokenizer 与模型


In [ ]:
# ============================================================
# Cell 9: 加载 Tokenizer
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(
    model_name_or_path, trust_remote_code=True,
    local_files_only=True)
print(f'bos_token_id: {tokenizer.bos_token_id}')
print(f'eos_token_id: {tokenizer.eos_token_id}')
print(f'pad_token_id: {tokenizer.pad_token_id}')
print(f'vocab_size:   {tokenizer.vocab_size}')
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
    print('pad_token 为 None，已设为 eos_token')
tokenizer.padding_side = 'right'
print(f'padding_side = {tokenizer.padding_side}')


In [ ]:
# ============================================================
# Cell 10: 加载模型 — 8×910B NPU 配置
# ============================================================
# Qwen3.5-9B BF16 ≈ 18GB，8×910B 显存充裕
base_model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    attn_implementation='sdpa',
    trust_remote_code=True,
    local_files_only=True)
print(f'模型加载完成')
print(f'模型参数量: {base_model.num_parameters():,}')
print(f'模型设备:   {base_model.device}')
# 检查词表大小
tsize = len(tokenizer)
vsize = base_model.config.vocab_size
if tsize != vsize:
    base_model.resize_token_embeddings(tsize)
    print(f'词表不一致，已 resize: {vsize} -> {tsize}')


In [ ]:
# ============================================================
# Cell 11: Gradient Checkpointing + 梯度配置
# ============================================================
# 必须与 SFTConfig 中的 gradient_checkpointing 保持一致
base_model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={'use_reentrant': False})
base_model.enable_input_require_grads()
base_model.config.use_cache = False
print('gradient_checkpointing: ON')
print('enable_input_require_grads: ON')
print('use_cache: OFF')


In [ ]:
# ============================================================
# Cell 12: 显存监控
# ============================================================
if torch.npu.is_available():
    for i in range(torch.npu.device_count()):
        free_mb, total_mb = [x/1024/1024 for x in torch.npu.mem_get_info(i)]
        used_mb = total_mb - free_mb
        print(f'NPU-{i}: 已用 {used_mb:.0f}MB / 总计 {total_mb:.0f}MB')


## Step 3: DeepSpeed 配置

DeepSpeed 版本必须锁定 0.16.9，0.17.4 不兼容昇腾 NPU


In [ ]:
# ============================================================
# Cell 13: 生成 DeepSpeed ZeRO2 配置文件
# ============================================================
ds_config = {
    'train_batch_size': 8,
    'gradient_accumulation_steps': 4,
    'bf16': {'enabled': True},
    'zero_optimization': {
        'stage': 2,
        'offload_optimizer': {'device': 'cpu'},
        'contiguous_gradients': True,
        'overlap_comm': True,
    },
    'gradient_clipping': 1.0,
    'steps_per_print': 10,
    'wall_clock_breakdown': False,
}
ds_config_path = os.path.join(path_project, 'ds_config.json')
with open(ds_config_path, 'w', encoding='utf-8') as f:
    json.dump(ds_config, f, indent=2)
print(f'DeepSpeed 配置已写入: {ds_config_path}')
print(json.dumps(ds_config, indent=2))


## Step 4: 训练参数配置


In [ ]:
# ============================================================
# Cell 14: SFTConfig 配置
# ============================================================
# 使用 TRL 的 SFTConfig，替代标准的 TrainingArguments
# packing=True — 将短序列拼接为长序列，减少 padding 浪费
# gradient_checkpointing 必须与 Cell 11 保持一致

sft_config = SFTConfig(
    output_dir=path_output,
    
    # packing —— SFTTrainer 特有，拼接短序列提升效率
    packing=True,
    max_seq_length=2048,
    dataset_text_field='text',
    
    # batch 配置
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    
    # 精度
    bf16=True,
    fp16=False,
    
    # 训练轮次
    num_train_epochs=3,
    
    # 梯度检查点（与 Cell 11 保持一致）
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    
    # 验证策略
    evaluation_strategy='steps',
    eval_steps=100,
    
    # 保存策略
    save_strategy='steps',
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='loss',
    greater_is_better=False,
    
    # 日志
    logging_steps=10,
    report_to='tensorboard',
    
    # DeepSpeed
    deepspeed=ds_config_path,
    ddp_find_unused_parameters=False,
    
    # 数据加载
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    
    # 学习率
    learning_rate=2e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    weight_decay=0.01,
    max_grad_norm=1.0,
    adam_beta1=0.9,
    adam_beta2=0.999,
)

global_batch = sft_config.per_device_train_batch_size * npu_cnt * sft_config.gradient_accumulation_steps
print(f'SFTConfig 配置完成')
print(f'每卡 batch: {sft_config.per_device_train_batch_size}')
print(f'全局 batch: {global_batch}')
print(f'训练轮次: {sft_config.num_train_epochs}')
print(f'学习率: {sft_config.learning_rate}')
print(f'packing: {sft_config.packing}')
print(f'max_seq_length: {sft_config.max_seq_length}')


## Step 5: 初始化 SFTTrainer 并训练


In [ ]:
# ============================================================
# Cell 15: 初始化 SFTTrainer
# ============================================================
# SFTTrainer 内置处理 packing，不需要 DataCollatorForSeq2Seq
# 启动命令：
#   FORCE_TORCHRUN=1 deepspeed --num_gpus 8 train.py

trainer = SFTTrainer(
    model=base_model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    dataset_text_field='text',  # SFTTrainer 用这个字段自动 tokenize
    max_seq_length=2048,
)

print(f'SFTTrainer 初始化完成')
print(f'训练集: {len(train_dataset)} 条')
print(f'验证集: {len(eval_dataset)} 条')


In [ ]:
# ============================================================
# Cell 16: 执行训练
# ============================================================
# 解释行运行：训练过程中 SFTTrainer 会自动处理 packing
# 如需断点续训，传 resume_from_checkpoint=True

# trainer.train()

# # 从 checkpoint 恢复：
# import glob
# ckpts = sorted(glob.glob(os.path.join(path_output, 'checkpoint-*')))
# trainer.train(resume_from_checkpoint=ckpts[-1])


## Step 6: 保存模型与推理测试


In [ ]:
# ============================================================
# Cell 17: 保存最终模型
# ============================================================
# 保存完整的 HF 格式模型（含 config.json / tokenizer 文件）
# trainer.save_model() 会保存 model 和 tokenizer

final_model_path = os.path.join(path_output, 'final_model')
os.makedirs(final_model_path, exist_ok=True)

# trainer.save_model(final_model_path)
# tokenizer.save_pretrained(final_model_path)

print('模型保存路径: ' + final_model_path)
print('取消注释上方代码后执行保存')


In [ ]:
# ============================================================
# Cell 18: 推理测试 — 验证训练效果
# ============================================================
# 加载训练好的模型并进行简单推理

# from transformers import AutoTokenizer, AutoModelForCausalLM
# import torch
#
# test_model = AutoModelForCausalLM.from_pretrained(
#     final_model_path, torch_dtype=torch.bfloat16,
#     device_map='auto', trust_remote_code=True)
# test_tokenizer = AutoTokenizer.from_pretrained(
#     final_model_path, trust_remote_code=True)
#
# messages = [
#     {'role': 'user', 'content': '请用一句话介绍南京。'}
# ]
# text = test_tokenizer.apply_chat_template(
#     messages, tokenize=False, add_generation_prompt=True)
# inputs = test_tokenizer(text, return_tensors='pt').to(test_model.device)
#
# with torch.no_grad():
#     outputs = test_model.generate(
#         **inputs, max_new_tokens=256,
#         temperature=0.7, top_p=0.9, do_sample=True)
# print(test_tokenizer.decode(outputs[0], skip_special_tokens=False))


附录：执行命令

将本 notebook 导出为 train.py 后，在 ModelArts 终端执行：

```bash
FORCE_TORCHRUN=1 deepspeed --num_gpus 8 train.py
```

单卡调试（不启用 DeepSpeed）：
```bash
python train.py
```
